# README
This notebook was used to explore and create the drafts of the performance comparison metric calculations.

It is not used in the S2CF+C approach, but could be used to more specifically and fine-grained explore the metrics on more specific data. Make sure to update file and folder paths and data column names.

For a reader that is not familiar with python code and would just like to execute and explore the S2CF+C approach or comparison module, I would suggest to look into the S2CF and comparison module, and the corresponding util files since they are more cleaned up and commented on.

In [ ]:
import pandas as pd
import os

VERSION = 14
SHOW = True

graphs_output_paths = [
    f'',
]

In [ ]:
# Activity Time comparison

# Collect all file data in one combined dataframe
dfs = []

input_folder_path = f''

endtime_file_paths = [
]

for file_path in endtime_file_paths:
  df = pd.read_csv(file_path)

  # Add metadata
  df["source_file"] = file_path.split('\\')[-1].replace('.csv', '').replace('_V2_annotated','')

  df["Datetime"] = pd.to_datetime(df["Datetime"], format="%d %b %Y %H:%M:%S,%f")
  df["End Datetime"] = pd.to_datetime(df["End Datetime"], format="%d %b %Y %H:%M:%S,%f")

  df["Duration"] = df["End Datetime"] - df["Datetime"]
  duration_seconds = df["Duration"].dt.total_seconds()
  df["Duration_s"] = duration_seconds

  dfs.append(df)

# Merge all
activity_durations_combined: pd.DataFrame = pd.concat(dfs, ignore_index=True)

# -----------------------------------------
# 1. Study-level totals
# -----------------------------------------

study_activity_totals: pd.DataFrame = (
    activity_durations_combined
    .groupby(["source_file", "Study ID", "Activity"], as_index=False)["Duration_s"]
    .sum()
    .rename(columns={"Duration_s": "Total_Duration_s"})
)

activity_summary: pd.DataFrame = (
    study_activity_totals
    .groupby(["source_file", "Activity"])["Total_Duration_s"]
    .agg(["min", "max", "mean", "median", "count"])
)

# -----------------------------------------
# 2. Event-level statistics
# -----------------------------------------
summary = activity_durations_combined.groupby(["source_file", "Activity"])["Duration_s"].agg(
  ["min", "max", "mean", "median", "count"]
)

# Ensure the output dirs exist
for path in graphs_output_paths:
  os.makedirs(path, exist_ok=True)

if SHOW:
  print("\n📊 Total duration per study and phase:\n")
  print(activity_summary)

  print("\n📌 Summary statistics per file and phase:\n")
  print(summary)

# Save summary table to ALL output paths
for path in graphs_output_paths:
  summary.to_csv(os.path.join(path, f"total-activity-duration_V{VERSION}.csv"))
  activity_summary.to_csv(os.path.join(path, f"avg-activity-duration_V{VERSION}.csv"))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import math
import os
import re

sns.set_theme(
    style="whitegrid",
    context="talk",
    font_scale=0.9
)

# ------------------------------------------------------------
# FIGURE A: FACETED BOXPLOTS AVERAGE ACTIVITY DURATION (INDEPENDENT Y-AXIS)
# ------------------------------------------------------------

unique_filtered_phases = [
    p for p in activity_durations_combined["Activity"].unique().tolist()
    if not re.match(r"^2_\d+", p)  # Exclude phases starting with "2_" followed by digits
]

print("Unique phases for plotting:", unique_filtered_phases)

cols = 2
rows = math.ceil(len(unique_filtered_phases) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(14, 4 * rows))
axes = axes.flatten()

dataset_order = sorted(activity_durations_combined["source_file"].unique())

for ax_i, phase in enumerate(unique_filtered_phases):
    ax = axes[ax_i]
    phase_data = activity_durations_combined[activity_durations_combined["Activity"] == phase].copy()

    # ---------- UNIT SELECTION ----------
    q95 = phase_data["Duration_s"].quantile(0.95)

    if q95 < 120:
        scale = 1
        unit = "seconds"
    elif q95 < 2 * 3600:
        scale = 60
        unit = "minutes"
    else:
        scale = 3600
        unit = "hours"

    phase_data["Duration_scaled"] = phase_data["Duration_s"] / scale

    # ---------- SEABORN BOXPLOT ----------
    sns.boxplot(
        data=phase_data,
        x="source_file",
        y="Duration_scaled",
        order=dataset_order,
        ax=ax,
        palette="Set2",
        hue="source_file",
        fliersize=5,
        linewidth=1.2
    )

    absolute_max = phase_data["Duration_scaled"].max()
    
    if not math.isnan(absolute_max) and absolute_max > 0:
        ax.set_ylim(0, absolute_max * 1.15)
    else:
        ax.set_ylim(0, 1)

    # ---------- MEDIAN ANNOTATIONS ----------
    medians = phase_data.groupby("source_file")["Duration_scaled"].median()
    
    for i, dataset in enumerate(dataset_order):
        if dataset in medians:
            median_val = medians[dataset]
            ax.text(
                i,
                median_val,
                f"{median_val:.2f}",
                ha="center",
                va="bottom",
                fontsize=8,
                weight='bold',
                color='black'
            )

    # ---------- LABELS & STYLING ----------
    ax.set_title(phase.replace("_", " "), fontsize=12, weight="semibold")
    ax.set_xlabel("Dataset")
    ax.set_ylabel(f"Duration ({unit})")
    
    # Strip unnecessary outer borders for a cleaner look
    sns.despine(ax=ax, left=False, bottom=False)

# Remove unused axes
for j in range(len(unique_filtered_phases), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle(
    "Average activity duration comparison",
    fontsize=16,
    weight="bold"
)
plt.tight_layout(rect=[0, 0, 1, 0.96])

# ---------- SAVE ----------
for path in graphs_output_paths:
    plt.savefig(
        os.path.join(
            path,
            f"boxplots_faceted_avg-activity-duration_V{VERSION}.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

if SHOW:
    plt.show()
else:
    plt.close()

In [ ]:

# Set a clean Seaborn style globally (optional but recommended)
sns.set_theme(
  style="whitegrid",
  context="talk",
  font_scale=0.9
)

# ------------------------------------------------------------
# FIGURE A: FACETED BOXPLOTS TOTAL ACTIVITY DURATION (STUDY-LEVEL DURATIONS)
# ------------------------------------------------------------
unique_filtered_phases = [
    p for p in study_activity_totals["Activity"].unique().tolist()
    if not re.match(r"^2_\d+", p)  # Exclude phases starting with "2_" followed by digits
]
cols = 2
rows = math.ceil(len(unique_filtered_phases) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(14, 4 * rows))
axes = axes.flatten()

# Ensure we map the x-axis order consistently across the loops
# Seaborn uses categorical ordering, so explicitly defining it avoids surprises
dataset_order = sorted(study_activity_totals["source_file"].unique())


for ax_i, phase in enumerate(sorted(unique_filtered_phases)):
    ax = axes[ax_i]

    phase_data = study_activity_totals[
        study_activity_totals["Activity"] == phase
    ].copy()

    # ---------- UNIT SELECTION (PER PHASE, STUDY-LEVEL) ----------
    q95 = phase_data["Total_Duration_s"].quantile(0.95)

    if q95 < 120:
        scale = 1
        unit = "seconds"
    elif q95 < 2 * 3600:
        scale = 60
        unit = "minutes"
    else:
        scale = 3600
        unit = "hours"

    phase_data["Duration_scaled"] = phase_data["Total_Duration_s"] / scale

    # ---------- SEABORN BOXPLOT ----------
    sns.boxplot(
        data=phase_data,
        x="source_file",
        y="Duration_scaled",
        order=dataset_order,
        ax=ax,
        palette="Set2",
        hue="source_file",
        fliersize=5,      
        linewidth=1.2     
    )

    # ---------- FIXED: ENTIRE DATA RANGE Y-SCALING ----------
    absolute_max = phase_data["Duration_scaled"].max()
    
    if not math.isnan(absolute_max) and absolute_max > 0:
        ax.set_ylim(0, absolute_max * 1.15)
    else:
        ax.set_ylim(0, 1)

    # ---------- MEDIAN ANNOTATIONS ----------
    medians = phase_data.groupby("source_file")["Duration_scaled"].median()
    
    for i, dataset in enumerate(dataset_order):
        if dataset in medians:
            median_val = medians[dataset]
            ax.text(
                i,
                median_val,
                f"{median_val:.2f}",
                ha="center",
                va="bottom",
                fontsize=8,
                weight='bold',
                color='black'
            )

    # ---------- LABELS & STYLING ----------
    ax.set_title(phase.replace("_", " "), fontsize=12, weight="semibold")
    ax.set_xlabel("Dataset")
    ax.set_ylabel(f"Total activity duration ({unit})")
    sns.despine(ax=ax, left=False, bottom=False)

# Remove unused axes
for j in range(len(unique_filtered_phases), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle(
    "Total activity duration comparison",
    fontsize=16,
    weight="bold"
)
plt.tight_layout(rect=[0, 0, 1, 0.96])

# ---------- SAVE ----------
for path in graphs_output_paths:
    plt.savefig(
        os.path.join(
            path,
            f"boxplots_faceted_total-activity-duration_V{VERSION}.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

if SHOW:
    plt.show()
else:
    plt.close()

In [ ]:
  
# ------------------------------------------------------------
# FIGURE B: FACETED BOXPLOTS Average activity duration (LOG-SCALE Y-AXIS)
# ------------------------------------------------------------

# Copy to avoid side effects
plot_df = activity_durations_combined.copy()

# Create readable phase label
plot_df["Activity_Label"] = plot_df["Activity"].str.replace("_", " ")

phase_order = unique_filtered_phases
phase_order = sorted([p.replace("_", " ") for p in phase_order])
phase_order = [p.replace("_", " ") for p in phase_order if not p.startswith("0")]

dataset_order = sorted(plot_df["source_file"].unique().tolist())

sns.set_theme(
  style="whitegrid",
  context="talk",      # larger fonts
  font_scale=0.9
)

plt.figure(figsize=(max(18, len(phase_order) * 1.2), 8))

ax = sns.boxplot(
  data=plot_df,
  x="Activity_Label",
  y="Duration_s",
  hue="source_file",
  order=phase_order,
  hue_order=dataset_order,
  palette="Set2",
  linewidth=1,
  fliersize=3
)

# Log-scale y-axis
ax.set_yscale("log")

# Labels
ax.set_ylabel("Duration (seconds, log scale)")
ax.set_xlabel("Activity")
ax.set_title("Average activity duration comparison", pad=20)

# Improve x-axis readability
ax.tick_params(axis="x", rotation=40)
ax.margins(x=0.05)

# Grid on y-axis only
ax.grid(True, which="both", axis="y", alpha=0.3)
ax.grid(False, axis="x")

ax.legend(
  title="Dataset",
  loc="upper left",
  bbox_to_anchor=(1.01, 1),
  frameon=True
)

plt.tight_layout(rect=[0, 0, 0.88, 1])

for path in graphs_output_paths:
  plt.savefig(
    os.path.join(
      path,
      f"boxplot_all_phases_avg-activity-duration_V{VERSION}.png"
    ),
    dpi=300,
    bbox_inches="tight"
  )

if SHOW:
    plt.show()
else:
    plt.close()

In [ ]:
# ------------------------------------------------------------
# FIGURE B: FACETED BOXPLOTS AVG ACTIVITY DURATION (LINEAR Y-AXIS IN HOURS)
# ------------------------------------------------------------

# Copy to avoid side effects
plot_df = activity_durations_combined.copy()

# Convert seconds to hours
plot_df["Duration_h"] = plot_df["Duration_s"] / 3600

# Create readable phase label
plot_df["Activity_Label"] = plot_df["Activity"].str.replace("_", " ")

# Explicit ordering
phase_order = unique_filtered_phases
phase_order = [ p for p in unique_filtered_phases if not re.match(r"^2_\d+", p)]

phase_order = sorted([p.replace("_", " ") for p in phase_order])

plot_df["Dataset_Label"] = (
    plot_df["source_file"]
    .str.replace("_", "-")
    .str.replace("-removed-V2", "", regex=False)
)

dataset_order = sorted(plot_df["Dataset_Label"].unique())

sns.set_theme(
    style="whitegrid",
    context="talk",
    font_scale=0.9
)

plt.figure(figsize=(max(18, len(phase_order) * 1.2), 8))

ax = sns.boxplot(
    data=plot_df,
    x="Activity_Label",
    y="Duration_h",
    hue="Dataset_Label",
    order=phase_order,
    hue_order=dataset_order,
    palette="Set2",
    linewidth=1,
    fliersize=3
)

# ---------- LINEAR Y-AXIS IN HOURS ----------
ax.set_ylabel("Duration (hours)")
ax.set_xlabel("Activity")
ax.set_title("Average activity duration comparison", pad=20)

# Force sensible baseline for linear scale
ax.set_ylim(bottom=0)

# Improve x-axis readability
ax.tick_params(axis="x", rotation=40)
ax.margins(x=0.05)

# Grid on y-axis only
ax.grid(True, axis="y", alpha=0.3)
ax.grid(False, axis="x")

ax.legend(
    title="Dataset",
    loc="upper left",
    bbox_to_anchor=(1.01, 1),
    frameon=True
)

plt.tight_layout(rect=[0, 0, 0.88, 1])

# ---------- SAVE ----------
for path in graphs_output_paths:
    plt.savefig(
        os.path.join(
            path,
            f"boxplot_all_phases_avg-activity-duration_linear_hours_V{VERSION}.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

if SHOW:
    plt.show()
else:
    plt.close()


In [ ]:

# -----------------------------------------
# 2. BOXPLOTS TO COMPARE AVG ACTIVITY DURATION
# -----------------------------------------
import seaborn as sns
import matplotlib.pyplot as plt

# Consistent dataset order & color palette
dataset_order = sorted(activity_durations_combined["source_file"].unique().tolist())

palette = dict(
    zip(dataset_order, sns.color_palette("Set2", len(dataset_order)))
)

phases_sorted = activity_durations_combined["Activity"].unique()

for phase in phases_sorted:
    phase_data = activity_durations_combined[activity_durations_combined["Activity"] == phase].copy()

    # ------------------------------------------------
    # AUTOMATIC UNIT SELECTION (95th percentile)
    # ------------------------------------------------
    q95 = phase_data["Duration_s"].quantile(0.95)

    if q95 < 120:
        scale = 1
        unit = "seconds"
    elif q95 < 2 * 3600:
        scale = 60
        unit = "minutes"
    else:
        scale = 3600
        unit = "hours"

    phase_data["Duration_scaled"] = phase_data["Duration_s"] / scale

    # ------------------------------------------------
    # BOXPLOT
    # ------------------------------------------------
    plt.figure(figsize=(9, 5))
    ax = sns.boxplot(
        data=phase_data,
        x="source_file",
        y="Duration_scaled",
        order=dataset_order,
        palette=palette,
        hue="source_file",
        showfliers=True
    )

    # ------------------------------------------------
    # CALCULATE Y-OFFSET FOR MEDIAN LABELS
    # ------------------------------------------------
    y_min, y_max = ax.get_ylim()
    y_offset = 0.04 * (y_max - y_min)  # 4% of axis range

    medians = (
        phase_data
        .groupby("source_file")["Duration_scaled"]
        .median()
        .reindex(dataset_order)
    )

    for i, (dataset, median) in enumerate(medians.items()):
        ax.text(
            i,
            median + y_offset,
            f"{median:.2f}",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
            color="black",
            zorder=10
        )

    # ------------------------------------------------
    # LABELS & STYLE
    # ------------------------------------------------
    ax.set_title(
        f"Average duration comparison – activity: {phase.replace('_', ' ')}",
        pad=15
    )
    ax.set_xlabel("Dataset")
    ax.set_ylabel(f"Duration ({unit})")

    ax.grid(True, axis="y", alpha=0.3)

    plt.tight_layout()

    # ------------------------------------------------
    # SAVE
    # ------------------------------------------------
    for path in graphs_output_paths:
        plt.savefig(
            os.path.join(
                path,
                f"boxplot_activity_{phase}_V{VERSION}.png"
            ),
            dpi=300,
            bbox_inches="tight"
        )

    if SHOW:
        plt.show()
    else:
        plt.close()

In [ ]:
# ============================================================
#  ACTIVITY TIME ANALYSIS + BEFORE/AFTER SANEKY-STYLE COMPARISON
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib.transforms as transforms
import itertools

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
# Exclude the legend to improve readability of the duration labels
INCLUDE_LEGEND = False
graphs_output_paths = [

]

# Ensure dirs exist
for path in graphs_output_paths:
    os.makedirs(path, exist_ok=True)

# ------------------------------------------------------------
# LOAD CSV FILES
# ------------------------------------------------------------
dfs = []

input_folder_path = (

)

endtime_file_paths = [
]

for file_path in endtime_file_paths:
    df = pd.read_csv(file_path)

    df["source_file"] = os.path.basename(file_path).replace(".csv", "")
    df["Datetime"] = pd.to_datetime(df["Datetime"], format="%d %b %Y %H:%M:%S,%f")
    df["End Datetime"] = pd.to_datetime(df["End Datetime"], format="%d %b %Y %H:%M:%S,%f")
    df["Duration_s"] = (df["End Datetime"] - df["Datetime"]).dt.total_seconds()

    dfs.append(df)

combined: pd.DataFrame = pd.concat(dfs, ignore_index=True)

# ------------------------------------------------------------
# SUMMARY TABLE
# ------------------------------------------------------------
summary: pd.DataFrame = combined.groupby(["source_file", "Activity"])["Duration_s"].agg(
    ["min", "max", "mean", "median", "count"]
)

for path in graphs_output_paths:
    summary.to_csv(os.path.join(path, f"summary_statistics_V{VERSION}.csv"))

if SHOW:
    display(summary)


# ============================================================
#   BEFORE / AFTER GROUPING
# ============================================================
def detect_group(name):
    name = name.lower()
    if "pre" in name:
        return "Before"
    if "trial" in name:
        return "After"
    return "Unknown"

combined["Group"] = combined["source_file"].apply(detect_group)

phase_order_inc_v2 = combined["Activity"].unique().tolist()
phase_order_exc_v2 = [
    p for p in combined["Activity"].unique().tolist()
    if not re.match(r"^2_\d+", p)
  ]

phase_summary_inc_v2 = (
    combined.groupby(["Group", "Activity"])["Duration_s"]
    .mean()
    .reset_index()
    .pivot(index="Activity", columns="Group", values="Duration_s")
    .reindex(phase_order_inc_v2)
    .fillna(0)
)

phase_summary_exc_v2 = (
    combined.groupby(["Group", "Activity"])["Duration_s"]
    .mean()
    .reset_index()
    .pivot(index="Activity", columns="Group", values="Duration_s")
    .reindex(phase_order_exc_v2)
    .fillna(0)
)

# Convert to minutes
phase_summary_min_inc_v2 = phase_summary_inc_v2 / 60.0
phase_summary_min_exc_v2 = phase_summary_exc_v2 / 60.0

if SHOW:
    print("\nActivity means (minutes):")
    display(phase_summary_min_inc_v2)
    display(phase_summary_min_exc_v2)

phases_inc_v2 = phase_summary_min_inc_v2.index.tolist()
before_values_inc_v2 = phase_summary_min_inc_v2["Before"].tolist()
after_values_inc_v2 = phase_summary_min_inc_v2["After"].tolist()

phases_exc_v2 = phase_summary_min_exc_v2.index.tolist()
before_values_exc_v2 = phase_summary_min_exc_v2["Before"].tolist()
after_values_exc_v2 = phase_summary_min_exc_v2["After"].tolist()

# ============================================================
#   HELPER — LABEL ONLY IF BAR > 2 CM IN LENGTH
# ============================================================
def bar_is_long_enough(bar, min_length_cm=0.7):
    min_length_in = min_length_cm / 2.54
    ax = bar.axes
    fig = ax.figure

    fig.canvas.draw()

    x0 = bar.get_x()
    x1 = x0 + bar.get_width()

    # Transform to display coords
    disp_x0 = ax.transData.transform((x0, 0))[0]
    disp_x1 = ax.transData.transform((x1, 0))[0]

    bar_length_in = abs(disp_x1 - disp_x0) / fig.dpi
    return bar_length_in >= min_length_in


# ============================================================
#   SANKEY-STYLE BEFORE/AFTER PLOT
# ============================================================
def draw_sankey_plot(
    phases,
    before_values,
    after_values,
    plot_name=f"phase_comparison_sankey_V{VERSION}.png"
):
    phase_colors = [
        "#6aa5ff",  # blue
        "#e38d8d",  # rose red
        "#5b7f23",  # green
        "#8d76ad",  # violet
    ]

    # If more phases exist, auto-extend colors

    palette = itertools.cycle(plt.cm.tab20.colors)
    while len(phase_colors) < len(phases):
        phase_colors.append(next(palette))

    fig, ax = plt.subplots(figsize=(18, 6), facecolor="white")
    ax.set_facecolor("white")

    # Remove axes
    for side in ["top","right","left","bottom"]:
        ax.spines[side].set_visible(False)
    ax.set_xticks([])
    ax.set_yticks([])

    # Draw stacked bars
    def draw_stacked(values, y_pos):
        centers = []
        left = 0
        bars = []
        for v, col in zip(values, phase_colors):
            bar = ax.barh(y_pos, v, left=left, color=col, height=0.3)
            bars.append(bar[0])
            centers.append(left + v/2)
            left += v
        return bars, centers

    offset = max(max(before_values), max(after_values)) * 0.03  # small spacing

    before_bars, before_centers = draw_stacked(before_values, y_pos=1)
    after_bars, after_centers   = draw_stacked(after_values,  y_pos=0)

    total_before = sum(before_values)
    total_after  = sum(after_values)

    ax.text(total_before, 1, f"{total_before:.1f} min",
            va='center', ha='left', fontsize=16, color='black')

    ax.text(total_after + offset, 0, f"{total_after:.1f} min",
            va='center', ha='left', fontsize=16, color='black')


    # Add labels only if readable
    for bar, val in zip(before_bars, before_values):
        if bar_is_long_enough(bar):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_y()+bar.get_height()/2,
                f"{val:.1f}", color="white", ha="center", va="center", fontsize=18)

    for bar, val in zip(after_bars, after_values):
        if bar_is_long_enough(bar):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_y()+bar.get_height()/2,
                    f"{val:.1f}", color="white", ha="center", va="center", fontsize=18)

    # Connector lines (start to start)
    for b_bar, a_bar in zip(before_bars, after_bars):
        x_before = b_bar.get_x()
        x_after  = a_bar.get_x()

        y_before = b_bar.get_y() + b_bar.get_height()/2
        y_after  = a_bar.get_y() + a_bar.get_height()/2

        ax.plot([x_before, x_after], [y_before, y_after],
                color="black", linewidth=1)

    # Labels
    before_y = 1+ 0.2   # slightly above before bar
    after_y  = 0+ 0.2   # slightly above after bar

    ax.text(12, before_y, "Pre-AI-addition (minutes)", color="black", fontsize=18,
        va="bottom", ha="left")

    ax.text(12, after_y,  "Trial-AI-addition (minutes)", color="black", fontsize=18,
        va="bottom", ha="left")

    ax.set_ylim(-3, 1.5)
    # Legend
    handles = [Rectangle((0,0),1,1,color=c) for c in phase_colors]
    if INCLUDE_LEGEND:
        ax.legend(handles, phases, fontsize=16, facecolor="white", edgecolor="black", labelcolor="black", loc="upper left", bbox_to_anchor=(1.25, 1))

    plt.tight_layout()

    # Save
    for path in graphs_output_paths:
        plt.savefig(os.path.join(path, plot_name),
                dpi=300, bbox_inches="tight")

    if SHOW:
        plt.show()
    else:
        plt.close()


draw_sankey_plot(
    phases=phases_inc_v2,
    before_values=before_values_inc_v2,
    after_values=after_values_inc_v2,
    plot_name=f"phase_comparison_sankey_V{VERSION}_inc_v2.png"
)

draw_sankey_plot(
    phases=phases_exc_v2,
    before_values=before_values_exc_v2,
    after_values=after_values_exc_v2,
    plot_name=f"phase_comparison_sankey_V{VERSION}_exc_v2.png"
)

In [ ]:
# ------------------------------------------------------------
# CREATE COMBINED BOXPLOT FIGURE (ONE PANEL PER PHASE)
# ------------------------------------------------------------

unique_phases = [
    p for p in combined["Activity"].unique().tolist()
    if not re.match(r"^2_\d+", p)
  ]
n_phases = len(unique_phases)

# You can choose layout
cols = 2                                # change to 1, 2, 3, or auto
rows = math.ceil(n_phases / cols)

fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))
axes = axes.flatten()  # flatten for easy indexing

for ax_i, phase in enumerate(unique_phases):
  phase_name = phase.replace('_', ' ')

  ax = axes[ax_i]

  phase_data = combined[combined["Activity"] == phase]

  # Create boxplot for this phase
  phase_data.boxplot(column="Duration_s", by="source_file", ax=ax)

  ax.set_title(f"{phase_name}", fontsize=12)
  ax.set_xlabel("Dataset")
  ax.set_ylabel("Duration (seconds)")
  ax.grid(alpha=0.3)

# Remove empty subplots if phases < rows*cols
for j in range(len(unique_phases), len(axes)):
  fig.delaxes(axes[j])

# Clean global suptitle
plt.suptitle("Duration Comparison per Activity", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.97])

# ------------------------------------------------------------
# SAVE TO ALL OUTPUT LOCATIONS
# ------------------------------------------------------------
for path in graphs_output_paths:
  plt.savefig(os.path.join(path,
              f"boxplots_all_phases_combined_V{VERSION}.png"),
              dpi=300, bbox_inches="tight")

# Show only if allowed
if SHOW:
  plt.show()
else:
  plt.close()


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import pingouin as pg

# --- 1. CONFIGURATION ---
# Replace these with your actual DataFrame and column names
dfs = []

input_folder_path = f''

endtime_file_paths = [
]

for file_path in endtime_file_paths:
  df = pd.read_csv(file_path)

  # Add metadata
  df["source_file"] = file_path.split('\\')[-1].replace('.csv', '').replace('_V2_annotated','')

  df["Datetime"] = pd.to_datetime(df["Datetime"], format="%d %b %Y %H:%M:%S,%f")
  df["End Datetime"] = pd.to_datetime(df["End Datetime"], format="%d %b %Y %H:%M:%S,%f")

  df["Duration"] = df["End Datetime"] - df["Datetime"]
  duration_seconds = df["Duration"].dt.total_seconds()
  df["Duration_s"] = duration_seconds

  dfs.append(df)

# Merge all
combined = pd.concat(dfs, ignore_index=True)

df = combined.copy()
duration_col = "Duration_s"
group_col = "source_file"
activity_col = "Activity"

# Identify the two distinct datasets
datasets = df[group_col].unique()
if len(datasets) != 2:
    raise ValueError(f"Expected exactly 2 datasets, found {len(datasets)}: {datasets}")

ds1, ds2 = sorted(datasets)
unique_activities = sorted(df[activity_col].unique())

significance_rows = []

def calculate_cliffs_delta(x, y):
  """
  Computes Cliff's delta effect size between two independent groups.
  Returns a value between -1.0 and 1.0.
  """
  x = np.asarray(x)
  y = np.asarray(y)
  n_x, n_y = len(x), len(y)
  
  # Broad-casting matrix comparison to find more/less pairs efficiently
  diff_matrix = x[:, None] - y
  
  pos = np.sum((diff_matrix > 0).astype(int))
  neg = np.sum((diff_matrix < 0).astype(int))
  
  return (pos - neg) / (n_x * n_y)

# --- 2. STATISTICAL ANALYSIS PER ACTIVITY ---
for activity in unique_activities:
    # Filter data for the specific activity
    act_data = df[df[activity_col] == activity]
    
    data_ds1 = act_data[act_data[group_col] == ds1][duration_col].dropna()
    data_ds2 = act_data[act_data[group_col] == ds2][duration_col].dropna()
    
    n1, n2 = len(data_ds1), len(data_ds2)
    
    # Skip if we don't have enough data points to compute statistics
    if n1 < 5 or n2 < 5:
        continue
        
    # Calculate Medians (Robust against process mining tail outliers)
    median_1 = data_ds1.median()
    median_2 = data_ds2.median()
    diff_seconds = median_1 - median_2
    
    # Mann-Whitney U test (Non-parametric test for skewed duration distributions)
    u_stat, p_val = stats.mannwhitneyu(data_ds1, data_ds2, alternative='two-sided')
    
    # Cliff's Delta for Effect Size (-1.0 to +1.0)
    # Interpretation: <0.147 (Negligible), <0.33 (Small), <0.474 (Medium), otherwise (Large)
    d_val =calculate_cliffs_delta(data_ds1, data_ds2)
    
    # Map effect size value to a standard academic qualitative descriptor
    abs_d = abs(d_val)
    if abs_d < 0.147:   effect_size_label = "Negligible"
    elif abs_d < 0.330:  effect_size_label = "Small"
    elif abs_d < 0.474:  effect_size_label = "Medium"
    else:               effect_size_label = "Large"
    
    # Formatting significance stars
    if p_val < 0.001:    stars = "***"
    elif p_val < 0.01:   stars = "**"
    elif p_val < 0.05:   stars = "*"
    else:                stars = "ns" # non-significant
    
    significance_rows.append({
        "Activity": activity.replace("_", " "),
        f"Median {ds1} (s)": round(median_1, 2),
        f"Median {ds2} (s)": round(median_2, 2),
        "Δ Median (s)": round(diff_seconds, 2),
        "p-value": f"{p_val:.4f}" if p_val >= 0.0001 else "<0.0001",
        "Sign.": stars,
        "Cliff's d": round(d_val, 3),
        "Effect Size": effect_size_label
    })

# --- 3. DISPLAY THE PROCESS MINING TABLE ---
results_df = pd.DataFrame(significance_rows)


# Apply styling to make it thesis-ready
styled_results = results_df.style.hide(axis="index").set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#f4f4f4'), ('font-weight', 'bold'), ('text-align', 'center')]},
    {'selector': 'td', 'props': [('text-align', 'center')]}
])

styled_results

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import pingouin as pg

# CALCULATE SIGNIFICANCE FOR TOTAL ACTIVITY DURATION

# --- 1. CONFIGURATION ---
dfs = []

input_folder_path = f''

endtime_file_paths = [
]

for file_path in endtime_file_paths:
  df = pd.read_csv(file_path)

  # Add metadata
  df["source_file"] = file_path.split('\\')[-1].replace('.csv', '').replace('_V2_annotated','')

  df["Datetime"] = pd.to_datetime(df["Datetime"], format="%d %b %Y %H:%M:%S,%f")
  df["End Datetime"] = pd.to_datetime(df["End Datetime"], format="%d %b %Y %H:%M:%S,%f")

  df["Duration"] = df["End Datetime"] - df["Datetime"]
  duration_seconds = df["Duration"].dt.total_seconds()
  df["Duration_s"] = duration_seconds

  dfs.append(df)

# Merge all
combined = pd.concat(dfs, ignore_index=True)

df = combined.copy()
duration_col = "Duration_s"
group_col = "source_file"
activity_col = "Activity"
case_col = "Study ID"

agg_df = (
    df.groupby([group_col, case_col, activity_col])[duration_col]
      .sum()
      .reset_index()
)

# Replace df with aggregated version for analysis
df = agg_df.copy()
# Identify the two distinct datasets
datasets = df[group_col].unique()
if len(datasets) != 2:
    raise ValueError(f"Expected exactly 2 datasets, found {len(datasets)}: {datasets}")

ds1, ds2 = sorted(datasets)
unique_activities = sorted(df[activity_col].unique())

significance_rows = []

def calculate_cliffs_delta(x, y):
  """
  Computes Cliff's delta effect size between two independent groups.
  Returns a value between -1.0 and 1.0.
  """
  x = np.asarray(x)
  y = np.asarray(y)
  n_x, n_y = len(x), len(y)
  

  diff_matrix = x[:, None] - y
  
  pos = np.sum((diff_matrix > 0).astype(int))
  neg = np.sum((diff_matrix < 0).astype(int))
  
  return (pos - neg) / (n_x * n_y)

# --- 2. STATISTICAL ANALYSIS PER ACTIVITY ---
for activity in unique_activities:
    # Filter data for the specific activity
    act_data = df[df[activity_col] == activity]
    
    data_ds1 = act_data[act_data[group_col] == ds1][duration_col].dropna()
    data_ds2 = act_data[act_data[group_col] == ds2][duration_col].dropna()
    
    n1, n2 = len(data_ds1), len(data_ds2)
    
    # Skip if we don't have enough data points to compute statistics
    if n1 < 5 or n2 < 5:
        continue
        
    # Calculate Medians (Robust against process mining tail outliers)
    median_1 = data_ds1.median()
    median_2 = data_ds2.median()
    diff_seconds = median_1 - median_2
    
    # Mann-Whitney U test (Non-parametric test for skewed duration distributions)
    u_stat, p_val = stats.mannwhitneyu(data_ds1, data_ds2, alternative='two-sided')
    
    # Cliff's Delta for Effect Size (-1.0 to +1.0)
    # Interpretation: <0.147 (Negligible), <0.33 (Small), <0.474 (Medium), otherwise (Large)
    d_val =calculate_cliffs_delta(data_ds1, data_ds2)
    
    # Map effect size value to a standard academic qualitative descriptor
    abs_d = abs(d_val)
    if abs_d < 0.147:   effect_size_label = "Negligible"
    elif abs_d < 0.330:  effect_size_label = "Small"
    elif abs_d < 0.474:  effect_size_label = "Medium"
    else:               effect_size_label = "Large"
    
    # Formatting significance stars
    if p_val < 0.001:    stars = "***"
    elif p_val < 0.01:   stars = "**"
    elif p_val < 0.05:   stars = "*"
    else:                stars = "ns" # non-significant
    
    significance_rows.append({
        "Activity": activity.replace("_", " "),
        f"Median {ds1} (s)": round(median_1, 2),
        f"Median {ds2} (s)": round(median_2, 2),
        "Δ Median (s)": round(diff_seconds, 2),
        "p-value": f"{p_val:.4f}" if p_val >= 0.0001 else "<0.0001",
        "Sign.": stars,
        "Cliff's d": round(d_val, 3),
        "Effect Size": effect_size_label
    })

# --- 3. DISPLAY THE PROCESS MINING TABLE ---
results_df = pd.DataFrame(significance_rows)

print(results_df.to_latex(index=False))

# Apply styling to make it thesis-ready
styled_results = results_df.style.hide(axis="index").set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#f4f4f4'), ('font-weight', 'bold'), ('text-align', 'center')]},
    {'selector': 'td', 'props': [('text-align', 'center')]}
])

styled_results